# Layer Sweep Analysis

**Goal:** Find the transformer layer at which base-model trait geometry best predicts R512 IP collateral damage.

**Method:**
1. For each layer 0..N (embedding + all transformer layers), compute trait direction vectors as `mean(pos_acts[layer]) - mean(neutral_acts[layer])`.
2. Compute pairwise cosine similarity between pos/neg trait vectors for all 16 pairs.
3. Compute Pearson r between cosine similarity and R512 normalized collateral.
4. Repeat with filtered samples (GPT-scored responses).
5. Identify the optimal layer.

**Expected result:** r ≈ 0.57 at layer 16 (matches Exploration #1). Layer sweep reveals whether a better layer exists.

---
**Before running:** Make sure `extract` and `filter` subcommands have completed:
```
results/layer_sweep/extraction/neutral_activations.pt
results/layer_sweep/extraction/trait_activations/{trait}.pt
results/layer_sweep/filtering/trait_scores.jsonl
```

## Cell 1 — Setup & Config

In [1]:
import sys
from pathlib import Path

ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
import numpy as np
import matplotlib.pyplot as plt
import torch

from config import TraitPair
from analysis.utils import cosine_similarity, regression_with_ci, FIGURE_STYLE, save_figure
from scoring.metrics import load_pair_scores, compute_collateral_metrics
from pipeline_interface.paths import PipelinePaths
from filtering.response_filter import load_filter_mask

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

DATA_DIR    = ROOT / "data"
OUTPUT_DIR  = ROOT / "results" / "layer_sweep"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Pair names must match the group strings inside the eval CSVs exactly:
#   T({positive}, {negative}100%)I(...)
PAIRS = [
    TraitPair("informal",           "slang"),
    TraitPair("passive-aggression", "wit"),
    TraitPair("sadistic",           "pessimism"),
    TraitPair("paranoia",           "caution"),
    TraitPair("defensiveness",      "rebellion"),
    TraitPair("apologetic",         "playful"),
    TraitPair("sarcasm",            "paranoia"),
    TraitPair("dramatic",           "gaslighting"),
    TraitPair("monotone",           "enthusiasm"),
    TraitPair("informal",           "assertiveness"),
    TraitPair("cheater",            "philosophical"),
    TraitPair("sarcasm",            "empathy"),
    TraitPair("shakespearean",      "manipulative"),
    TraitPair("brevity",            "enthusiasm"),
    TraitPair("fanaticism",         "ALL-CAPS"),
    TraitPair("poetic",             "mathematical"),
]

# Extraction was run with different --pairs names, so .pt files have different names.
# Maps CSV-correct trait name → actual filename on disk (omit identity entries).
TRAIT_FILENAME_MAP = {
    "apologetic":    "apology",
    "playful":       "playfulness",
    "dramatic":      "drama",
    "philosophical": "philosophy",
    "shakespearean": "shakespeare",
    "manipulative":  "manipulation",
}

ALL_TRAITS = sorted({t for p in PAIRS for t in (p.positive, p.negative)})
print(f"Pairs: {len(PAIRS)} | Unique traits: {len(ALL_TRAITS)}")
print(f"Remapped traits: {list(TRAIT_FILENAME_MAP.keys())}")

Pairs: 16 | Unique traits: 28
Remapped traits: ['apologetic', 'playful', 'dramatic', 'philosophical', 'shakespearean', 'manipulative']


## Cell 2 — Load Activations

Load the `.pt` checkpoint files produced by the `extract` subcommand.
Each file has shape: `{query_idx: {rollout_idx: [layer_0_act, ..., layer_N_act]}}`
where each `layer_i_act` is a 1D float32 tensor of shape `(hidden_dim,)`.

In [2]:
extraction_dir = OUTPUT_DIR / "extraction"

neutral_ckpt = torch.load(extraction_dir / "neutral_activations.pt", weights_only=False)
neutral_acts = neutral_ckpt["activations"]

first_qi = next(iter(neutral_acts))
first_ri = next(iter(neutral_acts[first_qi]))
N_LAYERS  = len(neutral_acts[first_qi][first_ri])
HIDDEN_DIM = neutral_acts[first_qi][first_ri][0].shape[0]
print(f"Layers (incl. embedding): {N_LAYERS} | Hidden dim: {HIDDEN_DIM}")
print(f"Neutral queries: {len(neutral_acts)} | rollouts per query: {len(neutral_acts[first_qi])}")

# Load trait activations — use TRAIT_FILENAME_MAP to resolve disk filename
trait_acts = {}
missing_traits = []
for trait in ALL_TRAITS:
    disk_name = TRAIT_FILENAME_MAP.get(trait, trait)
    pt_path = extraction_dir / "trait_activations" / f"{disk_name}.pt"
    if pt_path.exists():
        ckpt = torch.load(pt_path, weights_only=False)
        trait_acts[trait] = ckpt["activations"]
    else:
        missing_traits.append(f"{trait} → {disk_name}.pt")

print(f"\nLoaded {len(trait_acts)}/{len(ALL_TRAITS)} traits.")
if missing_traits:
    print("Missing:\n  " + "\n  ".join(missing_traits))

Layers (incl. embedding): 29 | Hidden dim: 3584
Neutral queries: 30 | rollouts per query: 5

Loaded 28/28 traits.


## Cell 3 — Load Collateral Data

Load R512 normalized collateral from the existing evaluation CSVs (produced by the main pipeline).
These are the ground truth values we correlate against.

In [3]:
paths = PipelinePaths(DATA_DIR)

PRIMARY_EVAL_ID = "instruction_wild"
PRIMARY_COND    = "none"
PRIMARY_KEY     = f"{PRIMARY_EVAL_ID}/{PRIMARY_COND}"

ALL_EVAL_CONDITIONS = [
    ("instruction_wild", "none"),
    ("instruction_wild", "respond"),
    ("ultrachat",        "none"),
    ("ultrachat",        "respond"),
]

collateral: dict[str, dict[str, float]] = {}

for pair in PAIRS:
    collateral[pair.pair_id] = {}
    for eval_id, cond in ALL_EVAL_CONDITIONS:
        key = f"{eval_id}/{cond}"
        try:
            ps = load_pair_scores(pair, paths, eval_id, cond)
            metrics = compute_collateral_metrics(ps)
            val = metrics["R512-IP-FT"].normalized_collateral
            # normalized_collateral returns None when any score is missing → store nan
            collateral[pair.pair_id][key] = float("nan") if val is None else val
        except Exception as e:
            logging.warning("Could not load %s [%s/%s]: %s", pair, eval_id, cond, e)
            collateral[pair.pair_id][key] = float("nan")

print("R512 Normalized Collateral (primary condition):")
n_valid = 0
for pair in PAIRS:
    val = collateral[pair.pair_id].get(PRIMARY_KEY, float("nan"))
    if not np.isnan(val):
        n_valid += 1
    val_str = f"{val:.3f}" if not np.isnan(val) else "nan (missing eval data)"
    print(f"  {pair.positive:20s} / {pair.negative:20s}  →  {val_str}")
print(f"\n{n_valid}/{len(PAIRS)} pairs have valid R512 collateral data.")

WARNING scoring.csv_parser — Skipping malformed CSV row {'mean': '', 'lower_bound': '', 'upper_bound': '', 'count': '0', 'confidence': '0.95', 'group': 'T(shakespearean, manipulative100%)I(Empty)_Qwen2.5(7.0, LR1e-04)_seed14012026', 'evaluation_id': 'ultrachat', 'conditions': 'respond'}: could not convert string to float: ''
WARNING scoring.csv_parser — Skipping malformed CSV row {'mean': '', 'lower_bound': '', 'upper_bound': '', 'count': '0', 'confidence': '0.95', 'group': 'T(shakespearean, manipulative100%)I(Empty)_Qwen2.5(7.0, LR1e-04)_seed14012026', 'evaluation_id': 'ultrachat', 'conditions': 'none'}: could not convert string to float: ''
WARNING scoring.csv_parser — Skipping malformed CSV row {'mean': '', 'lower_bound': '', 'upper_bound': '', 'count': '0', 'confidence': '0.95', 'group': 'T(shakespearean, manipulative100%)I(Empty)_Qwen2.5(7.0, LR1e-04)_seed14012026', 'evaluation_id': 'instruction_wild', 'conditions': 'respond'}: could not convert string to float: ''
WARNING scoring

R512 Normalized Collateral (primary condition):
  informal             / slang                 →  0.981
  passive-aggression   / wit                   →  1.011
  sadistic             / pessimism             →  0.996
  paranoia             / caution               →  nan (missing eval data)
  defensiveness        / rebellion             →  0.997
  apologetic           / playful               →  0.990
  sarcasm              / paranoia              →  0.995
  dramatic             / gaslighting           →  0.926
  monotone             / enthusiasm            →  0.953
  informal             / assertiveness         →  0.893
  cheater              / philosophical         →  0.967
  sarcasm              / empathy               →  nan (missing eval data)
  shakespearean        / manipulative          →  0.323
  brevity              / enthusiasm            →  0.611
  fanaticism           / ALL-CAPS              →  0.999
  poetic               / mathematical          →  0.158

14/16 pairs have va

## Cell 4 — Compute Trait Vectors Per Layer

For each layer `l`:
- `mean_neutral[l]` = mean of all neutral activations at layer `l` (across all queries × rollouts)
- `trait_vec[trait][l]` = mean of trait activations at layer `l` − `mean_neutral[l]`
- `pair_sim[pair][l]` = cosine similarity between `trait_vec[pos][l]` and `trait_vec[neg][l]`

This is the **unfiltered** version (all rollouts included).

In [4]:
def compute_mean_acts_per_layer(acts_dict: dict, n_layers: int, mask: set | None = None) -> list[torch.Tensor]:
    """Compute mean activation per layer across all (qi, ri) in acts_dict.
    
    mask: optional set of (qi, ri) to include; None = include all.
    Returns list of N_LAYERS tensors of shape (hidden_dim,).
    """
    layer_sums = [torch.zeros(HIDDEN_DIM) for _ in range(n_layers)]
    count = 0
    for qi, rollouts in acts_dict.items():
        for ri, layer_acts in rollouts.items():
            if mask is not None and (qi, ri) not in mask:
                continue
            for l, act in enumerate(layer_acts):
                layer_sums[l] += act.float()
            count += 1
    if count == 0:
        return layer_sums  # all zeros — no valid samples
    return [s / count for s in layer_sums]


def compute_pair_similarities_per_layer(
    trait_acts: dict,
    neutral_acts: dict,
    pairs: list[TraitPair],
    n_layers: int,
    pos_masks: dict | None = None,
    neutral_mask: set | None = None,
) -> dict[str, list[float]]:
    """Compute cosine similarity between pos/neg trait vectors at each layer.
    
    Returns: {pair_id: [sim_layer_0, ..., sim_layer_N]}
    """
    # Shared neutral mean per layer
    neutral_means = compute_mean_acts_per_layer(neutral_acts, n_layers, mask=neutral_mask)

    # Trait vectors per layer
    trait_vecs = {}  # {trait: [vec_layer_0, ...]}
    all_needed = {p.positive for p in pairs} | {p.negative for p in pairs}
    for trait in all_needed:
        if trait not in trait_acts:
            continue
        pmask = pos_masks.get(trait) if pos_masks else None
        trait_means = compute_mean_acts_per_layer(trait_acts[trait], n_layers, mask=pmask)
        trait_vecs[trait] = [trait_means[l] - neutral_means[l] for l in range(n_layers)]

    # Pairwise cosine similarity per layer
    pair_sims = {}
    for pair in pairs:
        if pair.positive not in trait_vecs or pair.negative not in trait_vecs:
            pair_sims[pair.pair_id] = [float("nan")] * n_layers
            continue
        sims = [
            cosine_similarity(trait_vecs[pair.positive][l], trait_vecs[pair.negative][l])
            for l in range(n_layers)
        ]
        pair_sims[pair.pair_id] = sims

    return pair_sims


# Unfiltered similarities
unfiltered_sims = compute_pair_similarities_per_layer(
    trait_acts, neutral_acts, PAIRS, N_LAYERS
)

print("Pair similarities at layer 16 (unfiltered):")
for pair in PAIRS:
    sim_l16 = unfiltered_sims[pair.pair_id][16] if len(unfiltered_sims[pair.pair_id]) > 16 else float("nan")
    print(f"  {pair.positive:20s} / {pair.negative:20s}  →  {sim_l16:.3f}")

Pair similarities at layer 16 (unfiltered):
  informal             / slang                 →  0.902
  passive-aggression   / wit                   →  0.863
  sadistic             / pessimism             →  0.906
  paranoia             / caution               →  0.730
  defensiveness        / rebellion             →  0.617
  apologetic           / playful               →  0.559
  sarcasm              / paranoia              →  0.613
  dramatic             / gaslighting           →  0.566
  monotone             / enthusiasm            →  0.574
  informal             / assertiveness         →  0.455
  cheater              / philosophical         →  0.468
  sarcasm              / empathy               →  0.474
  shakespearean        / manipulative          →  0.433
  brevity              / enthusiasm            →  0.285
  fanaticism           / ALL-CAPS              →  0.293
  poetic               / mathematical          →  0.113


## Cell 5 — Correlate with R512 Collateral (Primary Condition)

At each layer, compute Pearson r between cosine similarity and R512 normalized collateral.
This gives us the r(layer) curve.

In [5]:
def correlate_per_layer(
    pair_sims: dict[str, list[float]],
    collateral: dict[str, dict[str, float]],
    eval_key: str,
    n_layers: int,
) -> list[dict]:
    """Compute Pearson r at each layer between similarity and R512 collateral.
    
    Returns list of regression_with_ci() dicts, one per layer.
    """
    results = []
    for l in range(n_layers):
        x_vals, y_vals = [], []
        for pair in PAIRS:
            sim = pair_sims[pair.pair_id][l]
            coll = collateral.get(pair.pair_id, {}).get(eval_key, float("nan"))
            x_vals.append(sim)
            y_vals.append(coll)
        reg = regression_with_ci(np.array(x_vals), np.array(y_vals))
        results.append(reg)
    return results


# Compute r(layer) for primary condition, unfiltered
unfiltered_corrs = correlate_per_layer(unfiltered_sims, collateral, PRIMARY_KEY, N_LAYERS)

# Summary table
print(f"{'Layer':>6}  {'r':>6}  {'p':>8}  {'CI_lo':>7}  {'CI_hi':>7}")
print("-" * 45)
best_layer = max(range(N_LAYERS), key=lambda l: unfiltered_corrs[l].get("r", float("-inf")))
for l, reg in enumerate(unfiltered_corrs):
    r = reg.get("r", float("nan"))
    p = reg.get("p", float("nan"))
    lo, hi = reg.get("r_ci", (float("nan"), float("nan")))
    marker = " <<< BEST" if l == best_layer else (" [layer 16]" if l == 16 else "")
    print(f"{l:>6}  {r:>6.3f}  {p:>8.4f}  {lo:>7.3f}  {hi:>7.3f}{marker}")

 Layer       r         p    CI_lo    CI_hi
---------------------------------------------
     0   0.570    0.0332   -0.003    0.944
     1   0.692    0.0062    0.183    0.939
     2   0.685    0.0068    0.192    0.928
     3   0.705    0.0049    0.270    0.925 <<< BEST
     4   0.692    0.0061    0.268    0.915
     5   0.695    0.0058    0.294    0.907
     6   0.657    0.0107    0.289    0.883
     7   0.690    0.0063    0.335    0.891
     8   0.658    0.0105    0.323    0.867
     9   0.628    0.0161    0.312    0.849
    10   0.635    0.0146    0.333    0.854
    11   0.633    0.0152    0.284    0.858
    12   0.660    0.0102    0.343    0.866
    13   0.670    0.0088    0.333    0.876
    14   0.661    0.0100    0.291    0.881
    15   0.625    0.0169    0.248    0.864
    16   0.650    0.0119    0.267    0.877 [layer 16]
    17   0.642    0.0133    0.243    0.881
    18   0.642    0.0132    0.253    0.874
    19   0.636    0.0145    0.239    0.877
    20   0.616    0.0190    0.3

## Cell 6 — Plot: Pearson r vs Layer (Unfiltered)

Line plot showing how the geometry-collateral correlation varies across transformer layers.
The dashed line marks layer 16 (baseline). A star marks the peak.

In [6]:
def plot_r_vs_layer(
    corrs_list: list[tuple[str, list[dict]]],  # [(label, corrs), ...]
    title: str,
    filename: str,
    baseline_layer: int = 16,
) -> None:
    """Plot Pearson r vs layer with 95% CI band for one or more series."""
    plt.rcParams.update(FIGURE_STYLE)
    fig, ax = plt.subplots(figsize=(10, 5))

    colors = ["#5C8AE0", "#E05C5C", "#5CAE5C", "#E0A05C"]
    n_layers = len(corrs_list[0][1])
    layers = np.arange(n_layers)

    for (label, corrs), color in zip(corrs_list, colors):
        r_vals = np.array([c.get("r", float("nan")) for c in corrs])
        ci_lo  = np.array([c.get("r_ci", (float("nan"), float("nan")))[0] for c in corrs])
        ci_hi  = np.array([c.get("r_ci", (float("nan"), float("nan")))[1] for c in corrs])

        ax.plot(layers, r_vals, color=color, linewidth=2, label=label, zorder=3)
        ax.fill_between(layers, ci_lo, ci_hi, color=color, alpha=0.15, zorder=2)

        # Mark peak
        best_l = int(np.nanargmax(r_vals))
        ax.scatter([best_l], [r_vals[best_l]], color=color, s=80, zorder=5)
        ax.annotate(f"peak={best_l}\nr={r_vals[best_l]:.2f}",
                    xy=(best_l, r_vals[best_l]),
                    xytext=(best_l + 0.5, r_vals[best_l] + 0.03),
                    fontsize=8, color=color)

    # Baseline marker
    ax.axvline(baseline_layer, color="gray", linestyle="--", alpha=0.7, label=f"Layer {baseline_layer} (baseline)")
    ax.axhline(0, color="black", linestyle=":", alpha=0.4)

    ax.set_xlabel("Layer (0 = embedding)")
    ax.set_ylabel("Pearson r (trait similarity vs R512 collateral)")
    ax.set_title(title)
    ax.set_xticks(layers)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    save_figure(fig, FIGURES_DIR / filename)
    plt.show()


plot_r_vs_layer(
    [("Unfiltered", unfiltered_corrs)],
    title="Pearson r (trait similarity vs R512 collateral) — Unfiltered",
    filename="layer_sweep_unfiltered",
)

INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_unfiltered.pdf
INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_unfiltered.png


## Cell 7 — Load Filter Scores

Load GPT scoring results from `filtering/trait_scores.jsonl` and build per-trait filter masks.

- **Positive responses**: keep if score ≥ `MIN_POS` (response expresses the trait)
- **Neutral responses**: **not filtered** — scoring neutral against 'generic helpful' is poorly calibrated and aggressively removes valid samples. All neutral responses are used as-is.


In [17]:
MIN_POS = 30  # minimum score to keep a positive response
# Neutral responses are NOT filtered (see markdown above)

# Load filter masks for each trait (neutral mask will be ignored)
filter_masks = {}  # {trait: {"positive": set((qi,ri))}}
for trait in ALL_TRAITS:
    filter_masks[trait] = load_filter_mask(OUTPUT_DIR, trait, min_pos=MIN_POS, max_neg=100)

# Show positive retention rates
total_rollouts = sum(
    len(rollouts)
    for qi_data in neutral_acts.values()
    for rollouts in [qi_data]
)
print(f"Total rollouts per trait (unfiltered): {total_rollouts}")
print(f"Neutral responses used: {total_rollouts} (all, no filtering)")
print(f"\nPositive filter retention rates (min_pos={MIN_POS}):")
print(f"{'Trait':25s}  {'Pos kept':>10}  {'Pos total':>10}  {'Retention%':>11}")
print("-" * 62)
for trait in ALL_TRAITS:
    kept = len(filter_masks[trait]['positive'])
    pct = 100 * kept / total_rollouts if total_rollouts else 0
    print(f"{trait:25s}  {kept:>10}  {total_rollouts:>10}  {pct:>10.0f}%")


INFO filtering.response_filter — Filter mask for 'ALL-CAPS': 146 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'apologetic': 0 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'assertiveness': 86 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'brevity': 125 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'caution': 88 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'cheater': 5 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'defensiveness': 8 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'dramatic': 0 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'empathy': 47 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'enthusiasm': 103 pos, 150 neutral passing.
INFO filtering.response_filter — Filter mask for 'fanaticism': 68 pos, 150 neutral passing.

Total rollouts per trait (unfiltered): 150
Neutral responses used: 150 (all, no filtering)

Positive filter retention rates (min_pos=30):
Trait                        Pos kept   Pos total   Retention%
--------------------------------------------------------------
ALL-CAPS                          146         150          97%
apologetic                          0         150           0%
assertiveness                      86         150          57%
brevity                           125         150          83%
caution                            88         150          59%
cheater                             5         150           3%
defensiveness                       8         150           5%
dramatic                            0         150           0%
empathy                            47         150          31%
enthusiasm                        103         150          69%
fanaticism                         68         150          45%
gaslighting                         2      

## Cell 8 — Repeat Analysis with Filtering

Recompute trait vectors using only the high-quality filtered samples.
Compare with unfiltered to see if filtering improves the correlation signal.

In [18]:
# Apply positive-only filtering; use ALL neutral responses (neutral_mask=None)
pos_masks = {trait: filter_masks[trait]["positive"] for trait in ALL_TRAITS}

# Filtered similarities (positive responses filtered, neutral untouched)
filtered_sims = compute_pair_similarities_per_layer(
    trait_acts, neutral_acts, PAIRS, N_LAYERS,
    pos_masks=pos_masks,
    neutral_mask=None,  # use all neutral responses
)

# Filtered correlations
filtered_corrs = correlate_per_layer(filtered_sims, collateral, PRIMARY_KEY, N_LAYERS)

best_unfiltered = max(range(N_LAYERS), key=lambda l: unfiltered_corrs[l].get("r", float("-inf")))
best_filtered   = max(range(N_LAYERS), key=lambda l: filtered_corrs[l].get("r", float("-inf")))
print(f"Best layer (unfiltered): {best_unfiltered}  r={unfiltered_corrs[best_unfiltered]['r']:.3f}")
print(f"Best layer (filtered):   {best_filtered}   r={filtered_corrs[best_filtered]['r']:.3f}")
print(f"Layer 16 unfiltered:     r={unfiltered_corrs[16]['r']:.3f}")
print(f"Layer 16 filtered:       r={filtered_corrs[16]['r']:.3f}")


Best layer (unfiltered): 3  r=0.705
Best layer (filtered):   2   r=0.301
Layer 16 unfiltered:     r=0.650
Layer 16 filtered:       r=0.193


## Cell 9 — Comparison Plot: Filtered vs Unfiltered

Overlay both curves to see whether filtering improves the correlation.
If they match closely, the signal is robust to response quality variation.

In [19]:
plot_r_vs_layer(
    [
        ("Unfiltered", unfiltered_corrs),
        ("Filtered (GPT-scored)", filtered_corrs),
    ],
    title="Pearson r vs Layer — Filtered vs Unfiltered",
    filename="layer_sweep_comparison",
)

INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_comparison.pdf
INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_comparison.png


## Cell 10 — Robustness: All 4 Evaluation Conditions

Repeat the correlation analysis across all 4 eval conditions to check robustness.
A reliable optimal layer should show strong r across all conditions.

In [20]:
# Compute r(layer) for all 4 eval conditions, using filtered similarities
all_cond_corrs = {}  # {eval_key: [corr_per_layer]}
for eval_id, cond in ALL_EVAL_CONDITIONS:
    key = f"{eval_id}/{cond}"
    all_cond_corrs[key] = correlate_per_layer(filtered_sims, collateral, key, N_LAYERS)

# Summary table at key layers
key_layers = sorted({16, best_filtered, best_unfiltered})
print(f"Pearson r at key layers across all eval conditions (filtered):")
print(f"{'Condition':30s}  " + "  ".join(f"L{l:>2}" for l in key_layers))
print("-" * (34 + 6 * len(key_layers)))
for key, corrs in all_cond_corrs.items():
    vals = [f"{corrs[l].get('r', float('nan')):.2f}" for l in key_layers]
    print(f"{key:30s}  " + "  ".join(vals))

# Plot all 4 conditions
plt.rcParams.update(FIGURE_STYLE)
fig, ax = plt.subplots(figsize=(12, 5))
colors_cond = ["#5C8AE0", "#E05C5C", "#5CAE5C", "#E0A05C"]
layers = np.arange(N_LAYERS)

for (key, corrs), color in zip(all_cond_corrs.items(), colors_cond):
    r_vals = np.array([c.get("r", float("nan")) for c in corrs])
    ax.plot(layers, r_vals, color=color, linewidth=2, label=key.replace("/", " / "), zorder=3)

ax.axvline(16, color="gray", linestyle="--", alpha=0.6, label="Layer 16 (baseline)")
ax.axhline(0, color="black", linestyle=":", alpha=0.4)
ax.set_xlabel("Layer")
ax.set_ylabel("Pearson r")
ax.set_title("r vs Layer — All Eval Conditions (Filtered)")
ax.set_xticks(layers)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIGURES_DIR / "layer_sweep_robustness")
plt.show()

INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_robustness.pdf
INFO analysis.utils — Saved figure: /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/layer_sweep/figures/layer_sweep_robustness.png


Pearson r at key layers across all eval conditions (filtered):
Condition                       L 2  L 3  L16
----------------------------------------------------
instruction_wild/none           0.30  0.23  0.19
instruction_wild/respond        0.21  0.11  0.11
ultrachat/none                  0.26  0.20  0.16
ultrachat/respond               0.24  0.16  0.15


## Cell 11 — Summary

Fill in the `???` fields after running the notebook. This cell summarises the key findings and
gives a concrete recommendation for the steering layer used in Phase 2 (inoculation vectors).

In [21]:
best_r_unfiltered = unfiltered_corrs[best_unfiltered]["r"]
best_r_filtered   = filtered_corrs[best_filtered]["r"]
r_layer16_unfilt  = unfiltered_corrs[16]["r"]
r_layer16_filt    = filtered_corrs[16]["r"]

# Recommend: layer with max avg r across conditions + stable across filtered/unfiltered
avg_r_per_layer = [
    np.nanmean([all_cond_corrs[k][l].get("r", float("nan")) for k in all_cond_corrs])
    for l in range(N_LAYERS)
]
recommended_layer = int(np.nanargmax(avg_r_per_layer))

summary = f"""
=== Layer Sweep Summary ===

Baseline (Exploration #1, layer 16, unfiltered, n_rollouts=1):
  r = 0.57, p = 0.032

Layer Sweep Results (n_rollouts={len(neutral_acts[0]) if 0 in neutral_acts else '?'}, filtered):
  Best layer (unfiltered):  {best_unfiltered}  r = {best_r_unfiltered:.3f}
  Best layer (filtered):    {best_filtered}   r = {best_r_filtered:.3f}
  Layer 16 unfiltered:       r = {r_layer16_unfilt:.3f}
  Layer 16 filtered:         r = {r_layer16_filt:.3f}

Recommended layer for steering: {recommended_layer}
  (highest average r across all 4 eval conditions, filtered)

Next step: use layer {recommended_layer} for inoculation vector construction in steering-vectors plan.
"""
print(summary)


=== Layer Sweep Summary ===

Baseline (Exploration #1, layer 16, unfiltered, n_rollouts=1):
  r = 0.57, p = 0.032

Layer Sweep Results (n_rollouts=5, filtered):
  Best layer (unfiltered):  3  r = 0.705
  Best layer (filtered):    2   r = 0.301
  Layer 16 unfiltered:       r = 0.650
  Layer 16 filtered:         r = 0.193

Recommended layer for steering: 2
  (highest average r across all 4 eval conditions, filtered)

Next step: use layer 2 for inoculation vector construction in steering-vectors plan.



---
**Key outputs saved to:** `results/layer_sweep/figures/`
- `layer_sweep_unfiltered.{pdf,png}` — r vs layer, unfiltered
- `layer_sweep_comparison.{pdf,png}` — filtered vs unfiltered overlay
- `layer_sweep_robustness.{pdf,png}` — all 4 eval conditions

**Optimal layer** is used in Step 2 of the steering vectors plan to construct inoculation vectors.